# Nhóm 10 – Dự đoán tái nhập viện ở bệnh nhân đái tháo đường (Diabetes 130-US)
**Notebook tổng hợp:** cả nhóm dùng file này để thống nhất cách hoạt động của từng task, **từ trên xuống dưới**. Các bước **nối bằng file**: mỗi ô đọc file đầu ra của bước trước trong `output/` và ghi file cho bước sau. Vì vậy mỗi ô chạy độc lập được (chỉ cần file đầu vào đã có), và xem được mọi file trung gian.

| Bước | Task | Người | File đầu vào | File đầu ra (`output/`) |
|---|---|---|---|---|
| 1.1 | Bỏ thuộc tính định danh | Dũng | `data/diabetic_data.csv` | `step1.csv`, `step1_patient_nbr.csv` |
| 1.2 | Thống kê khuyết, bỏ cột khuyết > 50% | Sơn | `step1.csv` | `step2.csv`, `missing_rate.csv` |
| 1.3 | Bỏ dòng khuyết < 2%, tạo nhãn, chia 80/20 | Chiến | `step2.csv`, `step1_patient_nbr.csv` | `train.csv`, `test.csv` |
| 2.1 | One-Hot (fit trên train) | Dũng | `train.csv`, `test.csv` | `X_train_onehot.npz`, `X_test_onehot.npz`, `onehot_columns.txt`, `y_train.csv`, `y_test.csv` |
| 2.2 | PCA 60 thành phần (ma trận tương quan) | Sơn | `X_*_onehot.npz`, `onehot_columns.txt` | `X_train_pca.npy`, `X_test_pca.npy`, `pca_eigenvalues.npy`, `pca_components.npy`, `pca_feature_names.txt` |
| 2.3 | Chuẩn hóa sau PCA, scree plot, top 5 vector riêng | Chiến | `X_*_pca.npy`, `pca_*` | `X_train_pca_scaled.npy`, `X_test_pca_scaled.npy`, `eigenvalues.png` |
| 3 | k-NN, Softmax / Cây, RF / MLP, Ensemble | Dũng / Sơn / Chiến | `X_*_pca(_scaled).npy`, `y_*.csv` | |
| 4 | Đánh giá, confusion matrix, ROC, class imbalance | Dũng / Sơn / Chiến | | |

**Quy ước chung:** không đổi tên file trong bảng khi chưa báo cả nhóm. Mọi bước học từ dữ liệu (One-Hot, PCA, chuẩn hóa) chỉ **fit trên train** rồi áp dụng sang test. `SEED = 42`.

## 0. Cài đặt
Clone **cả ba nhánh** vào `branches/Dung`, `branches/Son`, `branches/Chien`. Pipeline chạy trong `branches/Dung` (nơi có dữ liệu gốc). Hai nhánh còn lại dùng để **đối chiếu** với kết quả Sơn và Chiến đã đẩy lên (các ô "Check chéo").

In [ ]:
import os
REPO = "https://github.com/chiensondung/HCMUT-AnIntroductiontoMachineLearningandApplication-Project.git"
BRANCHES = ["Dung", "Son", "Chien"]
!rm -rf branches
for b in BRANCHES:
    !git clone -q -b {b} {REPO} branches/{b}
BR = os.path.abspath("branches")       # thư mục chứa 3 nhánh
%cd branches/Dung
!pip -q install -r requirements.txt

In [ ]:
# Tham số dùng chung. Đổi ở đây, không đổi rải rác trong các ô sau.
import os
SEED = 42
MISS_THRESHOLD = 0.50   # bỏ cột khuyết > 50%
ROW_THRESHOLD = 0.02    # bỏ dòng ở các cột khuyết < 2%
N_COMPONENTS = 60       # số thành phần PCA
OUT = "output"
os.makedirs(OUT, exist_ok=True)

def check(name, mine, theirs):
    print(("OK   " if mine == theirs else "LỆCH ") + f"{name}: notebook = {mine} | nhánh = {theirs}")

## Giai đoạn 1: Khảo sát và xử lý dữ liệu khuyết
### 1.1 Bỏ thuộc tính định danh *(Dũng)*
`data/diabetic_data.csv` → `output/step1.csv`, `output/step1_patient_nbr.csv`

`encounter_id`, `patient_nbr`, `payer_code` không mang ý nghĩa dự đoán. `patient_nbr` vẫn cần để chia tập theo bệnh nhân ở bước 1.3, nên được lưu riêng (cùng thứ tự dòng), không dùng làm đặc trưng.

In [ ]:
import pandas as pd

raw = pd.read_csv("data/diabetic_data.csv", na_values="?", low_memory=False)
print("Kích thước gốc:", raw.shape)            # (101766, 50)

ID_COLS = ["encounter_id", "patient_nbr", "payer_code"]
print(raw[ID_COLS].nunique().to_dict())
print("payer_code khuyết (%):", round(raw["payer_code"].isna().mean() * 100, 2))

df = raw.drop(columns=ID_COLS)
assert df.shape == (101766, 47) and not set(ID_COLS) & set(df.columns)

df.to_csv(f"{OUT}/step1.csv", index=False)
raw[["patient_nbr"]].to_csv(f"{OUT}/step1_patient_nbr.csv", index=False)
print("Đã lưu step1.csv:", df.shape)            # (101766, 47)

### 1.2 Thống kê khuyết, bỏ cột khuyết > 50% *(Sơn)*
`output/step1.csv` → `output/step2.csv`, `output/missing_rate.csv`

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUT}/step1.csv", low_memory=False)
miss = df.isna().mean()
miss_table = (miss[miss > 0].sort_values(ascending=False) * 100).round(2)
print(miss_table)

drop_cols = miss[miss > MISS_THRESHOLD].index.tolist()
df = df.drop(columns=drop_cols)
assert df.shape == (101766, 44)

df.to_csv(f"{OUT}/step2.csv", index=False)
miss_table.rename("ty_le_khuyet_pct").to_csv(f"{OUT}/missing_rate.csv")
print("Cột bị bỏ:", drop_cols, "| đã lưu step2.csv:", df.shape)   # (101766, 44)

### 1.3 Bỏ dòng khuyết < 2%, tạo nhãn, chia train/test 80/20 *(Chiến)*
`output/step2.csv`, `output/step1_patient_nbr.csv` → `output/train.csv`, `output/test.csv`

Nhãn đa lớp `y_multi` (`<30`, `>30`, `NO`) và nhị phân `y_bin` (1 = tái nhập viện). Chia 80/20 phân tầng theo `y_multi`, **không để một bệnh nhân ở cả hai tập** (nhóm theo `patient_nbr`).

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv(f"{OUT}/step2.csv", low_memory=False)
groups = pd.read_csv(f"{OUT}/step1_patient_nbr.csv")["patient_nbr"]
assert len(df) == len(groups)

miss = df.isna().mean()
low_miss_cols = miss[(miss > 0) & (miss < ROW_THRESHOLD)].index.tolist()
keep = df[low_miss_cols].notna().all(axis=1)
print("Cột khuyết < 2%:", low_miss_cols, "| bỏ", int((~keep).sum()), "dòng, còn", int(keep.sum()))
df, groups = df[keep].reset_index(drop=True), groups[keep].reset_index(drop=True)

df["y_multi"] = df.pop("readmitted")
df["y_bin"] = (df["y_multi"] != "NO").astype(int)

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr, te = next(splitter.split(df, df["y_multi"], groups))
assert not set(groups[tr]) & set(groups[te])    # không rò rỉ bệnh nhân

train, test = df.iloc[tr], df.iloc[te]
train.to_csv(f"{OUT}/train.csv", index=False)
test.to_csv(f"{OUT}/test.csv", index=False)
print("Đã lưu train.csv:", train.shape, "| test.csv:", test.shape)
for name, part in (("train", train), ("test", test)):
    print(name, part["y_multi"].value_counts(normalize=True).round(3).to_dict())

**Check chéo với Chiến** (`task_1-3-result` trên nhánh `Chien`): số dòng và phân bố nhãn phải bằng nhau.

In [ ]:
import pandas as pd
mine_tr = pd.read_csv(f"{OUT}/train.csv", usecols=["y_multi"])
mine_te = pd.read_csv(f"{OUT}/test.csv", usecols=["y_multi"])
ct = pd.read_csv(f"{BR}/Chien/task_1-3-result/train_task1_3.csv", usecols=["y_multi"])
ce = pd.read_csv(f"{BR}/Chien/task_1-3-result/test_task1_3.csv", usecols=["y_multi"])
check("số dòng train", len(mine_tr), len(ct)); check("số dòng test", len(mine_te), len(ce))
check("nhãn train", mine_tr["y_multi"].value_counts().to_dict(), ct["y_multi"].value_counts().to_dict())

## Giai đoạn 2: One-Hot, PCA, chuẩn hóa
### 2.1 One-Hot Encoding *(Dũng)*
`output/train.csv`, `output/test.csv` → `output/X_train_onehot.npz`, `output/X_test_onehot.npz`, `output/onehot_columns.txt`, `output/y_train.csv`, `output/y_test.csv`

Mã hóa mọi biến phân loại (gồm mã ICD-9 `diag_1..3` và 3 cột mã số `admission_type_id`, `discharge_disposition_id`, `admission_source_id`). Giá trị khuyết là một loại riêng `Missing`. **Fit trên train**, `handle_unknown="ignore"` cho các loại chỉ có ở test.

In [ ]:
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv(f"{OUT}/train.csv", low_memory=False)
test = pd.read_csv(f"{OUT}/test.csv", low_memory=False)
LABELS = ["y_multi", "y_bin"]
X_train_raw, X_test_raw = train.drop(columns=LABELS), test.drop(columns=LABELS)

ID_AS_CAT = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
cat_cols = X_train_raw.select_dtypes(exclude="number").columns.tolist() + ID_AS_CAT
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]
print("Biến phân loại:", len(cat_cols), "| biến số:", len(num_cols))

for part in (X_train_raw, X_test_raw):
    part[cat_cols] = part[cat_cols].astype("string").fillna("Missing")

enc = OneHotEncoder(sparse_output=True, dtype="float64", handle_unknown="ignore")
Xc_train = enc.fit_transform(X_train_raw[cat_cols])
Xc_test = enc.transform(X_test_raw[cat_cols])

X_train_oh = sparse.hstack([sparse.csr_matrix(X_train_raw[num_cols].to_numpy("float64")), Xc_train]).tocsr()
X_test_oh = sparse.hstack([sparse.csr_matrix(X_test_raw[num_cols].to_numpy("float64")), Xc_test]).tocsr()
oh_names = num_cols + enc.get_feature_names_out(cat_cols).tolist()
assert X_train_oh.shape[1] == len(oh_names) == X_test_oh.shape[1]

sparse.save_npz(f"{OUT}/X_train_onehot.npz", X_train_oh)
sparse.save_npz(f"{OUT}/X_test_onehot.npz", X_test_oh)
open(f"{OUT}/onehot_columns.txt", "w").write("\n".join(oh_names))
train[LABELS].to_csv(f"{OUT}/y_train.csv", index=False)
test[LABELS].to_csv(f"{OUT}/y_test.csv", index=False)
print("Đã lưu One-Hot:", X_train_oh.shape, X_test_oh.shape)
for d in ("diag_1", "diag_2", "diag_3"):
    print(d, "->", sum(n.startswith(d + "_") for n in oh_names), "cột")

### 2.2 PCA 60 thành phần trên ma trận tương quan *(Sơn)*
`output/X_*_onehot.npz`, `output/onehot_columns.txt` → `output/X_train_pca.npy`, `output/X_test_pca.npy`, `output/pca_eigenvalues.npy`, `output/pca_components.npy`, `output/pca_feature_names.txt`

Tự cài đặt: ma trận tương quan của train → trị riêng/vector riêng → chiếu lên 60 vector riêng lớn nhất. Cột phương sai 0 trên train bị bỏ trước. Tính trực tiếp từ ma trận thưa.

In [ ]:
import numpy as np
from scipy import sparse

X_tr = sparse.load_npz(f"{OUT}/X_train_onehot.npz")
X_te = sparse.load_npz(f"{OUT}/X_test_onehot.npz")
oh_names = open(f"{OUT}/onehot_columns.txt").read().split("\n")

n = X_tr.shape[0]
mu = np.asarray(X_tr.mean(axis=0)).ravel()
var = np.asarray(X_tr.multiply(X_tr).mean(axis=0)).ravel() - mu**2
keep_cols = var > 1e-12
X_tr, X_te = X_tr[:, keep_cols], X_te[:, keep_cols]
mu, sigma = mu[keep_cols], np.sqrt(var[keep_cols])
pca_names = [c for c, k in zip(oh_names, keep_cols) if k]
print("Bỏ", int((~keep_cols).sum()), "cột phương sai 0 | còn", X_tr.shape[1], "chiều")

cov = (X_tr.T @ X_tr).toarray() / n - np.outer(mu, mu)
corr = cov / np.outer(sigma, sigma)
corr = (corr + corr.T) / 2

eigvals, eigvecs = np.linalg.eigh(corr)
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = np.clip(eigvals[order], 0, None), eigvecs[:, order]
V = eigvecs[:, :N_COMPONENTS]

def project(X):  # ((X - mu) / sigma) @ V, không làm dày X
    return X @ (V / sigma[:, None]) - (mu / sigma) @ V

X_train_pca, X_test_pca = project(X_tr), project(X_te)
np.save(f"{OUT}/X_train_pca.npy", X_train_pca)
np.save(f"{OUT}/X_test_pca.npy", X_test_pca)
np.save(f"{OUT}/pca_eigenvalues.npy", eigvals)
np.save(f"{OUT}/pca_components.npy", V)
open(f"{OUT}/pca_feature_names.txt", "w").write("\n".join(pca_names))
print("Đã lưu PCA:", X_train_pca.shape, X_test_pca.shape,
      "| phương sai giải thích:", round(eigvals[:N_COMPONENTS].sum() / eigvals.sum(), 4))

**Check chéo với Sơn** (`output/` trên nhánh `Son`): kích thước ma trận PCA và tỉ lệ phương sai giải thích phải khớp.

In [ ]:
import numpy as np, pandas as pd
mine = np.load(f"{OUT}/X_train_pca.npy", mmap_mode="r")
ev = np.load(f"{OUT}/pca_eigenvalues.npy")
son = np.load(f"{BR}/Son/output/X_train_pca.npy", mmap_mode="r")
son_var = pd.read_csv(f"{BR}/Son/output/pca_explained_variance.csv")["ty_le_luy_tich"].iloc[-1]
check("kích thước train PCA", mine.shape, son.shape)
check("phương sai giải thích (60 thành phần)", round(float(ev[:N_COMPONENTS].sum() / ev.sum()), 4), round(float(son_var), 4))

### 2.3 Chuẩn hóa sau PCA, biểu đồ Eigenvalues, top 5 vector riêng *(Chiến)*
`output/X_*_pca.npy`, `output/pca_eigenvalues.npy`, `output/pca_components.npy`, `output/pca_feature_names.txt` → `output/X_train_pca_scaled.npy`, `output/X_test_pca_scaled.npy`, `output/eigenvalues.png`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

X_train_pca = np.load(f"{OUT}/X_train_pca.npy")
X_test_pca = np.load(f"{OUT}/X_test_pca.npy")
eigvals = np.load(f"{OUT}/pca_eigenvalues.npy")
V = np.load(f"{OUT}/pca_components.npy")
pca_names = open(f"{OUT}/pca_feature_names.txt").read().split("\n")

scaler = StandardScaler().fit(X_train_pca)           # fit trên train
X_train_sc, X_test_sc = scaler.transform(X_train_pca), scaler.transform(X_test_pca)
np.save(f"{OUT}/X_train_pca_scaled.npy", X_train_sc)
np.save(f"{OUT}/X_test_pca_scaled.npy", X_test_sc)
print("Độ lệch chuẩn trước (min, max):", X_train_pca.std(0).min().round(2), X_train_pca.std(0).max().round(2))
print("Độ lệch chuẩn sau  (min, max):", X_train_sc.std(0).min().round(2), X_train_sc.std(0).max().round(2))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(range(1, 101), eigvals[:100], marker="o", ms=3); ax[0].set(title="Eigenvalues (top 100)", xlabel="Thành phần", ylabel="Trị riêng")
ax[1].plot(range(1, N_COMPONENTS + 1), np.cumsum(eigvals[:N_COMPONENTS]) / eigvals.sum()); ax[1].set(title="Phương sai tích lũy", xlabel="Thành phần")
plt.tight_layout(); plt.savefig(f"{OUT}/eigenvalues.png", dpi=150); plt.show()

for k in range(5):  # 5 đặc trưng gốc đóng góp lớn nhất cho mỗi vector riêng
    top = np.argsort(np.abs(V[:, k]))[::-1][:5]
    print(f"PC{k + 1} (trị riêng {eigvals[k]:.2f}):", [(pca_names[i], round(float(V[i, k]), 3)) for i in top])

## Giai đoạn 3: Huấn luyện mô hình — chưa làm

## Giai đoạn 4: Đánh giá tổng hợp — chưa làm